In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Step 1: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/quora-insincere-questions-classification/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/quora-insincere-questions-classification/test.csv')

## Step 2: EDA

In [ ]:
# Create word count feature
train_df['word_count'] = train_df['question_text'].apply(lambda x: len(str(x).split()))

In [ ]:
from sklearn.model_selection import train_test_split

# features and target split
X = train_df['question_text']
y = train_df['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

print(f"Train set shape: {X_train.shape} | Insincere count: {y_train.sum()} ({y_train.mean():.2%})")
print(f"Val set shape:   {X_val.shape}  | Insincere count: {y_val.sum()} ({y_val.mean():.2%})")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),        # Captures single words and 2-word phrases
    min_df=5,                  # Drops ultra-rare tokens and typos
    max_features=50000,        # Keeps the vocab size manageable
    stop_words='english'       # Removes starndard noise words
)

# fit on training data and transform both sets
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print(f"Vocab size: {len(vectorizer.vocabulary_)}")
print(f"X_train_tfidf shape: {X_train_tfidf.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)

y_val_proba = model.predict_proba(X_val_tfidf)[:, 1]

In [ ]:
from sklearn.metrics import f1_score, classification_report

def find_best_threshold(y_true, y_proba):
    """
    Sweeps thresholds from 0.01 to 0.99 to find the cutoff that maximizes the F1-score.
    Returns the best threshold and its corresponding score.
    """
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_thresh = 0.5
    best_f1 = 0.0

    for thresh in thresholds:
        # Convert probabilities to binary predictions based on the current threshold
        y_pred = (y_proba >= thresh).astype(int)
        score = f1_score(y_true, y_pred)

        if score > best_f1:
            best_f1 = score
            best_thresh = thresh

    print(f"Best Threshold: {best_thresh:.2f}")
    print(f"Maximum F1-Score: {best_f1:.4f}\n")

    return best_thresh, best_f1

# 5. Run the sweep on validation predictions
best_threshold, max_f1 = find_best_threshold(y_val, y_val_proba)

# Evaluate the final tuned model setup
final_preds = (y_val_proba >= best_threshold).astype(int)
print("--- Final Classification Report ---")
print(classification_report(y_val, final_preds))

## Step 3: Submit and check

In [ ]:
X_test_tfidf = vectorizer.transform(test_df['question_text'])

test_probabilities = model.predict_proba(X_test_tfidf)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

submission_df = pd.DataFrame({
    'qid': test_df['qid'],
    'prediction': test_predictions
})

submission_df.to_csv('submission.csv', index=False)

## Step 4: Preprocessing/text cleaning

#### Build a vocabulary counter from the training text

In [ ]:
from collections import Counter

def build_vocab(texts):
    vocab = Counter()
    for text in texts:
        for word in text.split():
            vocab[word] += 1
    return vocab

vocab = build_vocab(X_train)
print(f"Number of unique words: {len(vocab)}")
print(vocab.most_common(10))

#### loading the embedding file

In [ ]:
import zipfile

zip_path = '/kaggle/input/competitions/quora-insincere-questions-classification/embeddings.zip'
extract_path = '/kaggle/working/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import shutil

# Target directories to wipe out entirely
folders_to_delete = [
    '/kaggle/working/wiki-news-300d-1M',
    '/kaggle/working/paragram_300_sl999',
    '/kaggle/working/GoogleNews-vectors-negative300'
]

for folder in folders_to_delete:
    if os.path.exists(folder):
        print(f"Deleting heavy directory: {folder}...")
        shutil.rmtree(folder)
        print(" -> Deleted successfully.")
    else:
        print(f"Directory not found (already cleared): {folder}")

print("\n--- Current Workspace Status ---")
print(os.listdir('/kaggle/working'))

In [ ]:
def load_embeddings(filepath):
    embeddings_index = {}
    fail_count = 0

    with open(filepath, encoding='utf-8', errors='ignore') as f:
        for line in f:
            try:
                values = line.rstrip().split(' ')

                # If a line splits into exactly 301 items, values[0] is the word,
                # values[1:] are the numbers
                if len(values) == 301:
                    word = values[0]
                    vector = np.asarray(values[1:], dtype='float32')
                    embeddings_index[word] = vector
                else:
                    # Catching lines where the word itself contains spaces (like glove.840B often does)
                    word = "".join(values[:-300])
                    vector = np.asarray(values[-300:], dtype='float32')
                    embeddings_index[word] = vector

            except Exception:
                fail_count += 1
                continue

    print(f"Loaded {len(embeddings_index)} word vectors")
    print(f"Failed lines: {fail_count}")
    return embeddings_index

# Full path to the text file inside the extracted folder
glove_path = '/kaggle/working/glove.840B.300d/glove.840B.300d.txt'
embeddings_index = load_embeddings(glove_path)

### coverage checking

In [ ]:
def check_coverage(vocab, embeddings_index):
    known_words = {}
    unknown_words = {}
    known_count = 0
    unknown_count = 0

    for word in vocab.keys():
        if word in embeddings_index:
            known_words[word] = embeddings_index[word]
            known_count += vocab[word]
        else:
            unknown_words[word] = vocab[word]
            unknown_count += vocab[word]

    vocab_coverage = len(known_words) / len(vocab)
    text_coverage = known_count / (known_count + unknown_count)

    print(f"Found embeddings for {vocab_coverage:.2%} of vocab")
    print(f"Found embeddings for {text_coverage:.2%} of all text")

    # sort unknown words by frequency, most common first
    unknown_words_sorted = sorted(unknown_words.items(), key=lambda x: x[1], reverse=True)

    return unknown_words_sorted

oov = check_coverage(vocab, embeddings_index)
oov[:20]

### Text cleaning

In [ ]:
import re

def clean_text(text):
    """
    Cleans raw text by isolating punctuation marks with spaces,
    allowing pre-trained embeddings to catch them as separate tokens.
    """
    text = str(text)

    # List of punctuation characters to isolate
    # Includes standard punctuation, mathematical symbols, and diverse quotes
    puncts = [
        ',', '.', '"', ':', ')', '(', '-', '!', '?', '|', ';', "'", '$', '&',
        '/', '[', ']', '>', '%', '=', '#', '*', '+', '\\', '•',  '~', '@', '£',
        '·', '_', '{', '}', '©', '^', '®', '`',  '→', '°', '€', '™', '›',  '♥',
        '←', '×', '§', '″', '′', 'Â', '█', '½', 'à', '…', '“', '★', '”', '–',
        '●', 'â', '►', '−', '¢', '²', '░', '¹', '◦', '°', '♦', 'ã', '³', '✦',
        'μ', 'ℹ', 'α', 'σ', '⇒', '❌', '👉', '˙', '⚙', '✈', '➡', '🇪🇸', '🍁', '〰',
        '📍', '😂', '🔥', '📌', '💖', '☑', '◣', '⛳', '🤠', '🤙', '🔵', '🏁', '✨'
    ]

    # 1. Isolate every punctuation mark with spaces on either side
    for p in puncts:
        if p in text:
            text = text.replace(p, f' {p} ')

    # 2. Standardize consecutive whitespaces into a single clean space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# 1. Apply the clean_text function to the dataset
cleaned_questions = train_df['question_text'].apply(clean_text)

# 2. Build the new vocabulary from the cleaned text
cleaned_vocab = build_vocab(cleaned_questions)

# 3. Check the updated coverage metrics
print("\n--- Coverage After Cleaning Punctuation ---")
oov_cleaned = check_coverage(cleaned_vocab, embeddings_index)

# 4. Preview the new top 20 OOV words
print("\n Top 20 OOV Words after cleaning:")
display(oov_cleaned[:20])

In [ ]:
import re
import operator
from tqdm import tqdm

# 1. COMPREHENSIVE CONTRACTION MAPPING
CONTRACTION_MAP = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot", "'cause": "because",
    "could've": "could have", "couldn't": "could not", "didn't": "did not",
    "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will", "he's": "he is",
    "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
    "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have",
    "I'm": "I am", "I've": "I have", "i'd": "i would", "i'll": "i will", "i'm": "i am",
    "i've": "i have", "isn't": "is not", "it'd": "it would", "it'll": "it will",
    "it's": "it is", "let's": "let us", "ma'am": "madam", "mayn't": "may not",
    "might've": "might have", "mightn't": "might not", "must've": "must have",
    "mustn't": "must not", "needn't": "need not", "oughtn't": "ought not",
    "shan't": "shall not", "sha'n't": "shall not", "she'd": "she would",
    "she'll": "she will", "she's": "she is", "should've": "should have",
    "shouldn't": "should not", "so've": "so have", "so's": "so is", "that'd": "that would",
    "that's": "that is", "there'd": "there would", "there's": "there is",
    "they'd": "they would", "they'll": "they will", "they're": "they are",
    "they've": "they have", "to've": "to have", "wasn't": "was not", "we'd": "we would",
    "we'll": "we will", "we're": "we are", "we've": "we have", "weren't": "were not",
    "what'll": "what will", "what're": "what are", "what's": "what is", "what've": "what have",
    "when's": "when is", "when've": "when have", "where'd": "where did", "where's": "where is",
    "where've": "where have", "who'll": "who will", "who's": "who is", "who've": "who have",
    "why's": "why is", "why've": "why have", "will've": "will have", "won't": "will not",
    "would've": "would have", "wouldn't": "would not", "y'all": "you all",
    "you'd": "you would", "you'll": "you will", "you're": "you are", "you've": "you have"
}

def clean_text_v2(text):
    text = str(text)

    # STEP 1: Normalize curly/variant quotes and apostrophes to straight ones
    # This ensures contractions match our dictionary keys perfectly
    text = text.replace("’", "'").replace("‘", "'").replace("´", "'").replace("`", "'")
    text = text.replace("“", '"').replace("”", '"').replace("„", '"')

    # STEP 2: Expand Contractions (Word-by-word comparison)
    # Using space-splitting temporary lookups so we don't accidentally match parts of regular words
    words = text.split()
    expanded_words = [CONTRACTION_MAP.get(word, word) for word in words]
    text = " ".join(expanded_words)

    # STEP 3: Isolate Punctuation
    puncts = [
        ',', '.', '"', ':', ')', '(', '-', '!', '?', '|', ';', "'", '$', '&',
        '/', '[', ']', '>', '%', '=', '#', '*', '+', '\\', '•',  '~', '@', '£',
        '·', '_', '{', '}', '©', '^', '®', '→', '°', '€', '™', '›',  '♥',
        '←', '×', '§', '″', '′', 'Â', '█', '½', 'à', '…', '–', '●', 'â',
        '►', '−', '¢', '²', '░', '¹', '◦', '♦', 'ã', '³', '✦', 'μ', 'ℹ',
        'α', 'σ', '⇒', '❌', '👉', '˙', '⚙', '✈', '➡', '😂', '🔥', '📌', '💖'
    ]
    for p in puncts:
        if p in text:
            text = text.replace(p, f' {p} ')

    # STEP 4: Collapse consecutive whitespaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# 1. Transform your dataset questions
v2_cleaned_questions = train_df['question_text'].apply(clean_text_v2)

# 2. Rebuild the vocabulary mapping
v2_vocab = build_vocab(v2_cleaned_questions)

# 3. Calculate updated embedding scores
print("\n--- Diagnostic Check: Coverage Post-Contraction Expansion ---")
oov_v2 = check_coverage(v2_vocab, embeddings_index)

# 4. Preview the remaining baseline OOV items
print("\nRemaining Top 20 OOV Words:")
display(oov_v2[:20])

## Step 5: building the embedding matrix and moving toward the LSTM

#### tokenization (converting text to integer sequences)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Prepare Cleaned Splits & Clean Test Data
X_train_clean, X_val_clean, y_train, y_val = train_test_split(
    v2_cleaned_questions,
    train_df['target'],
    test_size=0.1,
    stratify=train_df['target'],
    random_state=42
)

# Clean the test set questions using v2 cleaning as well
test_questions_clean = test_df['question_text'].apply(clean_text_v2)

MAX_VOCAB_SIZE = 50000
MAX_LEN = 50

# 1. Fit tokenizer on cleaned TRAINING text only
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
tokenizer.fit_on_texts(X_train_clean)

# 2. Convert cleaned text to integer sequences for train/val/test
train_sequences = tokenizer.texts_to_sequences(X_train_clean)
val_sequences = tokenizer.texts_to_sequences(X_val_clean)
test_sequences = tokenizer.texts_to_sequences(test_questions_clean)

# 3. Pad sequences to uniform length
X_train_pad = pad_sequences(train_sequences, maxlen=MAX_LEN, padding='pre', truncating='pre')
X_val_pad = pad_sequences(val_sequences, maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test_pad = pad_sequences(test_sequences, maxlen=MAX_LEN, padding='pre', truncating='pre')

print(f"X_train_pad shape: {X_train_pad.shape}")
print(f"X_val_pad shape: {X_val_pad.shape}")
print(f"X_test_pad shape: {X_test_pad.shape}")
print(f"Word index size: {len(tokenizer.word_index)}")

### Building the embedding matrix

In [ ]:
EMBED_DIM = 300

# 1. Decide matrix size
# index 0 is reserved for padding tokens, so we add 1 to our maximum vocabulary cutoff
vocab_size = MAX_VOCAB_SIZE + 1

# 2. Initialize with zeros
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))

# 3. Fill in vectors for known words
for word, i in tokenizer.word_index.items():
    # Skip words that fall outside the top 50,000 most frequent boundary
    if i >= MAX_VOCAB_SIZE:
        continue

    # Retrieve the pre-trained vector from GloVe
    embedding_vector = embeddings_index.get(word)

    if embedding_vector is not None:
        # Assign the pre-trained 300-dimension vector to its corresponding index row
        embedding_matrix[i] = embedding_vector

print(f"Embedding matrix shape: {embedding_matrix.shape}")

# 4. Sanity check: how much of the capped vocab actually got a real vector?
# Check along axis 1 (columns) to see which rows have a non-zero sum, then count them
nonzero_rows = np.count_nonzero(np.sum(embedding_matrix, axis=1) != 0)
print(f"Words with embeddings: {nonzero_rows} / {vocab_size} ({nonzero_rows/vocab_size:.2%})")

## Step 6: LSTM

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, GlobalMaxPooling1D, Dense, Dropout

# # Clear any background Keras sessions
# tf.keras.backend.clear_session()

# model = Sequential([
#     # Modern Keras entrance point: Tells the model to expect sentences of length 50
#     Input(shape=(MAX_LEN,), dtype='int32', name="input_layer"),

#     # 1. Embedding Layer: Plug in the pre-trained GloVe weights and freezing them
#     Embedding(
#         input_dim=vocab_size,
#         output_dim=EMBED_DIM,
#         weights=[embedding_matrix],
#         trainable=False,  # Protects the 15M pre-trained parameters from distortion
#         name="glove_embeddings"
#     ),

#     # 2. Recurrent Layer: Bidirectional LSTM
#     # return_sequences=True is mandatory so the next pooling layer can see every timestep
#     Bidirectional(
#         LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
#         name="bi_lstm"
#     ),

#     # 3. Pooling Layer
#     GlobalMaxPooling1D(name="global_max_pooling"),

#     Dropout(0.3, name="dropout_layer"),

#     Dense(32, activation='relu', name="dense_hidden"),

#     Dense(1, activation='sigmoid', name="output_layer")
# ])

# # 6. Compile with class-imbalance aware metrics
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
#     loss='binary_crossentropy',
#     metrics=[
#         'accuracy',
#         tf.keras.metrics.AUC(name='auc'),
#         tf.keras.metrics.Precision(name='precision'),
#         tf.keras.metrics.Recall(name='recall')
#     ]
# )

# model.summary()

In [ ]:
# from tensorflow.keras.models import load_model

# # 1. Update these paths to point to your new LSTM dataset
# LSTM_INPUT_MODEL_PATH = '/kaggle/input/datasets/mdshafayeturrahman/quora-lstm-model/lstm_model.keras'
# LSTM_WORKING_MODEL_PATH = '/kaggle/working/lstm_model.keras'

# # 2. Check and load
# if os.path.exists(LSTM_INPUT_MODEL_PATH):
#     print(f"Found pinned LSTM model in custom Kaggle Dataset! Loading: {LSTM_INPUT_MODEL_PATH}")
#     model = load_model(LSTM_INPUT_MODEL_PATH)
#     print("Model loaded successfully.")

# elif os.path.exists(LSTM_WORKING_MODEL_PATH):
#     print(f"Found saved LSTM model in working directory! Loading: {LSTM_WORKING_MODEL_PATH}")
#     model = load_model(LSTM_WORKING_MODEL_PATH)
#     print("Model loaded successfully.")

# else:
#     print("No saved LSTM model found anywhere. Starting 85-minute training loop...")
#     early_stop_lstm = EarlyStopping(
#         monitor='val_auc',
#         patience=3,
#         mode='max',
#         restore_best_weights=True
#     )

#     history = model.fit(
#         X_train_pad, y_train,
#         validation_data=(X_val_pad, y_val),
#         epochs=10,
#         batch_size=512,
#         class_weight=class_weight_dict,
#         callbacks=[early_stop_lstm]
#     )

#     # Save to working directory
#     model.save(LSTM_WORKING_MODEL_PATH)
#     print(f"Training complete. Model saved to {LSTM_WORKING_MODEL_PATH}")

In [ ]:
# from sklearn.metrics import classification_report, f1_score

# # 1. Get validation probabilities
# # model.predict returns a continuous value between 0.0 and 1.0 for each question
# print("Generating predictions on validation set...")
# y_val_proba_lstm = model.predict(X_val_pad, batch_size=512).squeeze()

# # 2. Reuse the find_best_threshold function
# # This runs the exact same threshold sweep logic used for the baseline
# print("\n--- Tuning Classification Threshold for LSTM ---")
# best_lstm_threshold, max_lstm_f1 = find_best_threshold(y_val, y_val_proba_lstm)

# # --- 3. Compare directly against baseline ---
# baseline_f1 = 0.6048
# f1_improvement = max_lstm_f1 - baseline_f1

# print("--- Performance Comparison ---")
# print(f"Baseline TF-IDF Logistic Regression F1-Score : {baseline_f1:.4f}")
# print(f"Bidirectional LSTM F1-Score                  : {max_lstm_f1:.4f}")
# print(f"Absolute F1-Score Change                     : {f1_improvement:+.4f}")
# print("----------------------------------------------\n")

# # --- 4. Final Classification Report ---
# final_lstm_preds = (y_val_proba_lstm >= best_lstm_threshold).astype(int)
# print("--- Final LSTM Classification Report ---")
# print(classification_report(y_val, final_lstm_preds))

### Submit the LSTM

In [ ]:
# # Predict on the already-tokenized/padded test set
# test_probabilities = model.predict(X_test_pad, batch_size=512).squeeze()

# # Apply the LSTM's tuned threshold
# test_predictions = (test_probabilities >= best_lstm_threshold).astype(int)

# submission_df = pd.DataFrame({
#     'qid': test_df['qid'],
#     'prediction': test_predictions
# })
# submission_df.to_csv('submission.csv', index=False)

## Step 7: GRU + Attention

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
import tensorflow.keras.backend as K

class AttentionPooling(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        # input_shape example: (None, 50, 128) -> (batch, timesteps, hidden_dim)
        # We need one weight per hidden_dim feature, to turn each timestep's
        # 128-dim vector into a single score.
        self.W = self.add_weight(
            name="attn_weight",
            shape=(input_shape[-1], 1),   # (hidden_dim, 1) mapping features down to a 1D score
            initializer="glorot_uniform",
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        # inputs shape example: (batch, 50, 128)

        # Step 1: score each timestep -> shape becomes (batch, 50) after squeeze
        scores = K.dot(inputs, self.W)          # (batch, 50, 128) @ (128, 1) -> (batch, 50, 1)
        scores = K.squeeze(scores, axis=-1)     # -> (batch, 50)

        # Step 2: softmax across timesteps so weights sum to 1 per question
        # Given shape (batch, timesteps), axis=-1 is the timestep axis (axis 1)
        weights = tf.nn.softmax(scores, axis=-1)

        # Step 3: weighted sum of the original inputs using these weights
        weights_expanded = K.expand_dims(weights, axis=-1)   # (batch, 50, 1) for broadcasting
        weighted_inputs = inputs * weights_expanded           # (batch, 50, 128)

        # Summing across the timesteps collapses axis 1 -> leaving (batch, hidden_dim)
        output = tf.reduce_sum(weighted_inputs, axis=1)

        return output

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Bidirectional, GRU, Dropout, Dense

tf.keras.backend.clear_session()

model_gru = Sequential([
    Input(shape=(MAX_LEN,), dtype='int32', name="input_layer"),

    Embedding(
        input_dim=vocab_size,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        trainable=False,
        name="glove_embeddings"
    ),

    Bidirectional(
        # Swapped LSTM for GRU with matching parameter style
        GRU(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        name="bi_gru"
    ),

    AttentionPooling(name="attention_pooling"),  # Dynamic weighted average layer

    Dropout(0.3, name="dropout_layer"),
    Dense(32, activation='relu', name="dense_hidden"),
    Dense(1, activation='sigmoid', name="output_layer")
])

model_gru.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

model_gru.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping

# 1. Re-calculate class weights (now safely independent of LSTM code)
unique_classes = np.unique(y_train)
computed_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train
)
class_weight_dict = dict(zip(unique_classes, computed_weights))

# 1. Define your custom dataset paths
GRU_INPUT_MODEL_PATH = '/kaggle/input/datasets/mdshafayeturrahman/quora-gru-attention-model/gru_attention_model.keras'
GRU_WORKING_MODEL_PATH = '/kaggle/working/gru_attention_model.keras'
# 2. Check and load
if os.path.exists(GRU_INPUT_MODEL_PATH):
    print(f"Found pinned GRU model in your Kaggle Dataset! Loading: {GRU_INPUT_MODEL_PATH}")
    model_gru = load_model(GRU_INPUT_MODEL_PATH, custom_objects={'AttentionPooling': AttentionPooling})
    print("Model loaded successfully.")

elif os.path.exists(GRU_WORKING_MODEL_PATH):
    print(f"Found saved GRU model in working directory! Loading: {GRU_WORKING_MODEL_PATH}")
    model_gru = load_model(GRU_WORKING_MODEL_PATH, custom_objects={'AttentionPooling': AttentionPooling})
    print("Model loaded successfully.")

else:
    print("No saved GRU model found. Training from scratch...")

    # Redefining early stopping to reset its internal validation history state
    early_stop_gru = EarlyStopping(
        monitor='val_auc',
        patience=3,
        mode='max',
        restore_best_weights=True
    )

    history_gru = model_gru.fit(
        X_train_pad, y_train,
        validation_data=(X_val_pad, y_val),
        epochs=10,
        batch_size=512,
        class_weight=class_weight_dict,
        callbacks=[early_stop_gru]
    )

    model_gru.save(GRU_WORKING_MODEL_PATH)
    print(f"Training complete. Saved to {GRU_WORKING_MODEL_PATH}")

In [ ]:
print("Generating predictions on validation set (GRU + Attention)...")
y_val_proba_gru = model_gru.predict(X_val_pad, batch_size=512).squeeze()

print("\n--- Tuning Classification Threshold for GRU + Attention ---")
best_gru_threshold, max_gru_f1 = find_best_threshold(y_val, y_val_proba_gru)

print("--- Performance Comparison ---")
print(f"Baseline TF-IDF Logistic Regression F1 : 0.6048")
print(f"Bidirectional LSTM F1                  : 0.6805")
print(f"Bidirectional GRU + Attention F1        : {max_gru_f1:.4f}")
print(f"Change vs LSTM                          : {max_gru_f1 - 0.6805:+.4f}")
print("----------------------------------------------\n")

final_gru_preds = (y_val_proba_gru >= best_gru_threshold).astype(int)
print("--- Final GRU + Attention Classification Report ---")
print(classification_report(y_val, final_gru_preds))

### Submiting the GRU model

In [ ]:
test_probabilities_gru = model_gru.predict(X_test_pad, batch_size=512).squeeze()
test_predictions_gru = (test_probabilities_gru >= best_gru_threshold).astype(int)

submission_df = pd.DataFrame({
    'qid': test_df['qid'],
    'prediction': test_predictions_gru
})
submission_df.to_csv('submission.csv', index=False)